## Semi-Supervised Learning

Most times, manually labeling a complete dataset can be very time consuming, and in
real life scenarios, time is money. 

We are aiming to partially delete the target feature in
different proportions for each class, and run active and pasive semi-supervised learning
processes. This experiment will help us understand how it performs with our samples.


Our set of samples is not as large as we would like it to be, but it will be useful to study
how to identify the most interesting samples to manually label and leave the rest to the
model, saving the time and effort it takes to fully supervise a dataset

In [16]:
import pandas
import numpy as np


In [17]:
data = pandas.read_csv("../data/updated_pollution_dataset.csv")

In [18]:
target_col = data.columns[-1]

distribution = (
    data[target_col]
    .value_counts(dropna=False)
    .rename_axis(target_col)
    .reset_index(name="count")
)

distribution["percentage"] = (distribution["count"] / len(data) * 100).round(2)
distribution

,Air Quality,count,percentage
0,Good,2000,40.0
1,Moderate,1500,30.0
2,Poor,1000,20.0
3,Hazardous,500,10.0


In [19]:
quality_map = {
    "Good": 3,
    "Moderate": 2,
    "Poor": 1,
    "Hazardous": 0
}

data[target_col] = data[target_col].map(quality_map)


data[target_col] = data[target_col].astype("int8")
data[[target_col]].head()


,Air Quality
0,2
1,2
2,2
3,3
4,3


* Target variable partial deletion

1. Random Deletion

In [20]:
label_ratio = 0.1 # 10% of the data will be labeled
RANDOM_STATE = 42

data["true_label"] = data[target_col] # true labels for evaluation



PASSIVE Semi-Supervision

We ignore the class distribution and pseudo-randomly mask target samples

In [21]:
def passive_mask(data, target, label_ratio=0.1, random_state=42):
    data = data.copy(deep=True)
    unlabeled_idx = data.sample(frac=1 - label_ratio, random_state=random_state).index
    data.loc[unlabeled_idx, target] = np.nan
    return data

In [22]:
def active_mask(data, target, label_ratio, random_state=42):
    
    data = data.copy(deep=True)

    unlabeled_idx = (
        data.groupby(target, group_keys=False)          # group by class
          .apply(lambda g: g.sample(                  # from each class
              frac=1 - label_ratio,                 # mask this fraction
              random_state=random_state
          ))
          .index
    )
    data.loc[unlabeled_idx, target] = np.nan
    return data

In [23]:
data_passive = passive_mask(data, target_col, label_ratio=label_ratio, random_state=RANDOM_STATE)
data_active  = active_mask(data,  target_col, label_ratio=label_ratio, random_state=RANDOM_STATE)

In [26]:
for name, data in [("Passive", data_passive), ("Active", data_active)]:
    labeled   = data[target_col].notna()
    print(f"\n{'─'*40}")
    print(f"{name} - Labeled: {labeled.sum()} | Unlabeled: {(~labeled).sum()}")
    print("Label distribution (labeled only):")
    print(data.loc[labeled, target_col].value_counts(normalize=True).sort_index())


────────────────────────────────────────
Passive - Labeled: 500 | Unlabeled: 4500
Label distribution (labeled only):
Air Quality
0.0    0.106
1.0    0.198
2.0    0.306
3.0    0.390
Name: proportion, dtype: float64

────────────────────────────────────────
Active - Labeled: 500 | Unlabeled: 4500
Label distribution (labeled only):
Air Quality
0.0    0.1
1.0    0.2
2.0    0.3
3.0    0.4
Name: proportion, dtype: float64


Quality map for help:

quality_map = {
    "Good": 3,
    "Moderate": 2,
    "Poor": 1,
    "Hazardous": 0
}

In [28]:
data_active.head()


,Temperature,Humidity,PM2.5,PM10,NO2,SO2,CO,Proximity_to_Industrial_Areas,Population_Density,Air Quality,true_label
0,29.8,59.1,5.2,17.9,18.9,9.2,1.72,6.3,319,NaN,2
1,28.3,75.6,2.3,12.2,30.8,9.7,1.64,6.0,611,NaN,2
2,23.1,74.7,26.7,33.8,24.4,12.6,1.63,5.2,619,NaN,2
3,27.1,39.1,6.1,6.3,13.5,5.3,1.15,11.1,551,NaN,3
4,26.5,70.7,6.9,16.0,21.9,5.6,1.01,12.7,303,NaN,3


In [29]:
data_passive.head()

,Temperature,Humidity,PM2.5,PM10,NO2,SO2,CO,Proximity_to_Industrial_Areas,Population_Density,Air Quality,true_label
0,29.8,59.1,5.2,17.9,18.9,9.2,1.72,6.3,319,NaN,2
1,28.3,75.6,2.3,12.2,30.8,9.7,1.64,6.0,611,NaN,2
2,23.1,74.7,26.7,33.8,24.4,12.6,1.63,5.2,619,NaN,2
3,27.1,39.1,6.1,6.3,13.5,5.3,1.15,11.1,551,NaN,3
4,26.5,70.7,6.9,16.0,21.9,5.6,1.01,12.7,303,3.0,3


data split

In [ ]:
# Labeled pool - train supervised component
labeled_data = data_active[data_active[target_col].notna()]

# Unlabeled pool - semi-supervised propagation / pseudo-labeling
unlabeled_data = data_active[data_active[target_col].isna()]

# Ground truth  → final evaluation (never touch during training!)
y_true_unlabeled = data_active.loc[unlabeled_data.index, "true_label"]